In [14]:
#!pip install vllm --upgrade
!pip install faiss-cpu sentence-transformers SPARQLWrapper accelerate
#!pip install mistralai --upgrade
!pip install transformers --upgrade
!pip install mistral-common --upgrade
!pip install mistralai


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached mistralai-1.12.3-py3-none-any.whl.metadata (33 kB)
Using cached mistralai-1.12.3-py3-none-any.whl (502 kB)



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from SPARQLWrapper import SPARQLWrapper, JSON

def overlap_chunks(chunks, element):
    index = chunks.index(element)
    if(index == 0):
        return f"{element}. {chunks[index + 1]}"
    elif(len(chunks) - 1 == index):
        return f"{chunks[index - 1]}. {element}."
    else:
        return f"{chunks[index - 1]}. {element}. {chunks[index + 1]}"

def load_countries():
    ehri_sparql_endpoint = SPARQLWrapper("https://lod.ehri-project-test.eu/sparql")
    ehri_sparql_endpoint.setReturnFormat(JSON)
    ehri_sparql_endpoint.setQuery("""PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX ehri: <http://lod.ehri-project-test.eu/ontology#>
PREFIX rico: <https://www.ica.org/standards/RiC/ontology#>
SELECT ?s ?o WHERE {
  {
    ?s a ehri:Country ;
      ?p ?o .
    FILTER (isLiteral(?o) && ?p != rico:name)
  }
}""")
    result = ehri_sparql_endpoint.queryAndConvert()
    chunks = [chunk for row in result["results"]["bindings"] for chunk in row["o"]["value"].split(". ")]
    return list(map(lambda x: overlap_chunks(chunks, x), chunks))

def load_institutions():
    ehri_sparql_endpoint = SPARQLWrapper("https://lod.ehri-project-test.eu/sparql")
    ehri_sparql_endpoint.setReturnFormat(JSON)
    ehri_sparql_endpoint.setQuery("""PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX ehri: <http://lod.ehri-project-test.eu/ontology#>
PREFIX rico: <https://www.ica.org/standards/RiC/ontology#>
SELECT * WHERE {
    ?s a ehri:Institution ;
      rico:name ?name ;
      rico:generalDescription ?description ;
      rico:history ?history ;
      ehri:findingAids ?findingAids ;
      ehri:generalContext ?generalContext ;
      rdfs:seeAlso ?website ;
      ^rico:isOrWasLocationOfAgent/rico:name ?country .
}""")
    result = ehri_sparql_endpoint.queryAndConvert()
    for row in result["results"]["bindings"]:
        yield f"""Institution name: {row["name"]["value"]}
General description: {row["description"]["value"]}
History: {row["history"]["value"]}
Finding Aids: {row["findingAids"]["value"]}
General Context: {row["generalContext"]["value"]}
Website: {row["website"]["value"]}
Country: {row["country"]["value"]}"""

def load_chunks():
    return load_countries() + list(load_institutions())

In [2]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")
chunks = load_chunks()

embeddings = model.encode(chunks)

dimension = embeddings[0].shape[0]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

faiss.write_index(index, "rag_index_ehri_countries_and_institutions.faiss")
with open("rag_chunks.txt", "w", encoding="utf-8") as f:
    f.write("\n\n".join(chunks))

C:\Users\Herminio\Git\ragEhri\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|████████████████████████████████████████████████████| 103/103 [00:00<00:00, 543.06it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
def retrieve_relevant_chunks(query, top_k=6):
    # index = faiss.read_index("rag_index_ehri_countries.faiss")
    # with open("rag_chunks.txt", "r") as f:
    #     chunks = json.load(f)

    query_vec = model.encode([query])
    distances, indices = index.search(query_vec, top_k)
    #print(distances)
    #print(indices)

    return [chunks[indices[0][i]] for i in range(len(indices[0]))] #if distances[0][i] < 0.94]

In [4]:
retrieve_relevant_chunks("What was the situation of Spain during the Holocaust?")

['During and after the Civil War, the regime was responsible for the murder of around 150,000 political opponents and the incarceration of several hundred thousands in concentration and slave labor camps. Under Franco’s rule, Spain stayed neutral throughout the Second World War, but did send volunteers, the so-called Blue Division, to fight alongside the Axis Powers against the Soviet Union. The Spanish regime was complicit in the persecution and deportation of nearly 15,000 Spanish Republican exiles from German-occupied France to concentration camps within the Reich',
 'An additional 13,000 Jews lived in Spanish Morocco without Spanish citizenship. During the Civil War, many Jews left Spain, fearing the Fascists and their Axis allies. Antisemitism was an inherent part of the Franco Regime',
 'However, EHRI is yet to determine the exact nature and significance of Holocaust-related holdings in these institutions, as well as those of the Archivo General de la Nación [General Archive of t

In [5]:
retrieve_relevant_chunks("Which archives contain Holocaust-relevant material in the Netherlands?")

['In the Netherlands**\r\n\r\nA considerable number of Holocaust-related sources and collections were created in the Netherlands and can still be found in the country’s archives today. NIOD Institute for War, Holocaust and Genocide Studies holds important Holocaust collections. A joint effort of the National Archives and NIOD led to the creation of a website on Dutch WWII archival sources',
 'There are also a number of private archives and documentation centres, such as the Jewish Historical Museum in Amsterdam, which hold important Holocaust-related collections.\r\nEHRI identified over 160 repositories in the Netherlands and is able to provide thousands of descriptions of Holocaust-relevant archival units, most of which can be found at NIOD and at the Amsterdam City Archives.\r\n\r\n**C.II. In other countries**\r\n\r\nEHRI has identified and partially described archival institutions and/or collections outside of the Netherlands that are relevant to Holocaust research on the Netherland

In [6]:
retrieve_relevant_chunks("Where is Beit Lohamei archive located?")

["Institution name: Beit Lohamei Haghetaot Archives/ בית לוחמי הגטאות\nGeneral description: <b>– on the Holocaust of the Jewish people and Jewish resistance</b><br>\r\nHISTORICAL CONTEXT OF THE MATERIALS: covers three main time periods:<br>\r\n(a) <u>Jewish communities in Europe between the two world wars</u>: youth movements; religious life; society, education and culture; photographs, photo negatives, films, video-recorded testimonies, audiotaped testimonies, photograph records, press clippings, and a great many documents: personal official papers, letters and postcards, administrative documents, handwritten testimonies and memoirs, diaries, literary works and musical compositions.<br>\r\n(b) <u> The fate of the Jewish people under the Nazi regime and during WWII </u>: increasing restrictions on civil and human rights; in camps and ghettos, forests and hiding places; physical and spiritual resistance; rescue and extermination; <br>\r\n(c) <u>The rebuilding of lives after the Liberati

In [211]:
# from mistralai import Mistral
# import os

# # SYSTEM_PROMPT = """You are Mistral Small 3.1, a Large Language Model (LLM) created by Mistral AI, a French startup headquartered in Paris.
# # You power an AI assistant called Le Chat.
# # Your knowledge base was last updated on 2023-10-01.
# # The current date is {today}.
# # You are now being used in a Retrieval Augmented Generation (RAG) set up using data from the EHRI Portal which will feed some contextual information to you under the header # CONTEXT
# # If this contextual information does not provide good answers just follow the general behaviour defined below.

# # When you're not sure about some information, you say that you don't have the information and don't make up anything.
# # If the user's question is not clear, ambiguous, or does not provide enough context for you to accurately answer the question, you do not try to answer it right away and you rather ask the user to clarify their request (e.g. "What are some good restaurants around me?" => "Where are you?" or "When is the next flight to Tokyo" => "Where do you travel from?").
# # You are always very attentive to dates, in particular you try to resolve dates (e.g. "yesterday" is {yesterday}) and when asked about information at specific dates, you discard information that is at another date.
# # You follow these instructions in all languages, and always respond to the user in the language they use or request.
# # Next sections describe the capabilities that you have."""

# def get_results_llm(user_prompt):
#     with Mistral(api_key=os.getenv("MISTRAL_API_KEY", ""),) as mistral:
#         return mistral.chat.complete(model="mistral-small-latest", 
#             messages=[
#                 # {
#                 #     "content": SYSTEM_PROMPT,
#                 #     "role": "system"
#                 # },
#                 {
#                     "content": user_prompt,
#                     "role": "user",
#                 },
#             ], stream=False).choices[0].message.content

# def get_results_llm_with_rag(user_prompt):
#     relevant_context = "\n\n".join(retrieve_relevant_chunks(user_prompt))
#     prompt_with_context = user_prompt + f"""\n\n
#     # CONTEXT
#     {relevant_context}
#     """
#     return get_results_llm(prompt_with_context)

In [20]:
## This is the code to load the LLM locally
# from vllm import LLM
# from vllm.sampling_params import SamplingParams
# from datetime import datetime, timedelta

# SYSTEM_PROMPT = """You are Mistral Small 3.1, a Large Language Model (LLM) created by Mistral AI, a French startup headquartered in Paris.
# You power an AI assistant called Le Chat.
# Your knowledge base was last updated on 2023-10-01.
# The current date is {today}.
# You are now being used in a Retrieval Augmented Generation (RAG) set up using data from the EHRI Portal which will feed some contextual information to you under the header # CONTEXT
# If this contextual information does not provide good answers just follow the general behaviour defined below.

# When you're not sure about some information, you say that you don't have the information and don't make up anything.
# If the user's question is not clear, ambiguous, or does not provide enough context for you to accurately answer the question, you do not try to answer it right away and you rather ask the user to clarify their request (e.g. "What are some good restaurants around me?" => "Where are you?" or "When is the next flight to Tokyo" => "Where do you travel from?").
# You are always very attentive to dates, in particular you try to resolve dates (e.g. "yesterday" is {yesterday}) and when asked about information at specific dates, you discard information that is at another date.
# You follow these instructions in all languages, and always respond to the user in the language they use or request.
# Next sections describe the capabilities that you have.

# # WEB BROWSING INSTRUCTIONS

# You cannot perform any web search or access internet to open URLs, links etc. If it seems like the user is expecting you to do so, you clarify the situation and ask the user to copy paste the text directly in the chat.

# # MULTI-MODAL INSTRUCTIONS

# You have the ability to read images, but you cannot generate images.
# You cannot read nor transcribe audio files or videos."""

# def get_results_llm(user_prompt):
#   messages = [
#       {
#           "role": "system",
#           "content": SYSTEM_PROMPT
#       },
#       {
#           "role": "user",
#           "content": user_prompt
#       },
#   ]
#   model_name = "mistralai/Mistral-Small-3.1-24B-Instruct-2503"
#   # note that running this model on GPU requires over 60 GB of GPU RAM
#   llm = LLM(model=model_name, tokenizer_mode="mistral")

#   sampling_params = SamplingParams(max_tokens=512, temperature=0.15)
#   outputs = llm.chat(messages, sampling_params=sampling_params)

#   return outputs[0].outputs[0].text

In [4]:
import torch
from transformers import Mistral3ForConditionalGeneration, MistralCommonBackend
from transformers import TextIteratorStreamer
from threading import Thread

def get_results_llm(user_prompt):
    model_id = "mistralai/Ministral-3-3B-Instruct-2512-BF16"

    tokenizer = MistralCommonBackend.from_pretrained(model_id)
    model = Mistral3ForConditionalGeneration.from_pretrained(model_id, device_map="auto")
    
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": user_prompt,
                }
            ],
        },
    ]

    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    
    tokenized = tokenizer.apply_chat_template(messages, return_tensors="pt", return_dict=True)
    
    # output = model.generate(
    #     **tokenized,
    #     max_new_tokens=512,
    #     streamer=streamer,
    # )[0]

    generation_kwargs = dict(tokenized, streamer=streamer, max_new_tokens=1024)

    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()
    
    #decoded_output = tokenizer.decode(output[len(tokenized["input_ids"][0]):])
    #return (decoded_output)
    return streamer

def get_results_llm_with_rag(user_prompt):
    relevant_context = "\n\n".join(retrieve_relevant_chunks(user_prompt))
    prompt_with_context = user_prompt + f"""\n\n
    # CONTEXT
    {relevant_context}
    """
    return get_results_llm(prompt_with_context)

def print_streamer(streamer):
    for new_text in streamer:
        print(new_text, end="")

In [5]:
#Without RAG
print_streamer(get_results_llm("What was the situation of Spain during the Holocaust?"))

Loading weights: 100%|███████████████| 458/458 [00:00<00:00, 472.80it/s, Materializing param=model.vision_tower.transformer.layers.23.ffn_norm.weight]
Some parameters are on the meta device because they were offloaded to the cpu and disk.


During the **Holocaust (1939–1945)**, Spain maintained a **neutral stance** in the conflict, avoiding direct involvement in Nazi Germany’s genocide against Jews and other persecuted groups. However, its position was complex due to its historical ties with fascist regimes, its own Jewish population, and its role in the broader geopolitical struggle. Here’s a breakdown of the key aspects:

### **1. Spain’s Official Neutrality (1939–1943)**
- After the **Spanish Civil War (1936–1939)**, which ended with the victory of **General Francisco Franco’s Nationalists** (backed by Nazi Germany and Fascist Italy), Spain declared itself a **neutral nation** in World War II.
- Unlike other European countries, Spain **did not join the Axis Powers** (Germany, Italy, Japan) or the Allies, though it maintained diplomatic relations with both sides.
- Franco’s regime was **pro-Nazi in ideology** (inspired by fascism) but **anti-communist**, making it a key ally of the Axis in Europe.

### **2. Spain’s Jewi

In [7]:
#With RAG
print_streamer(get_results_llm_with_rag("What was the situation of Spain during the Holocaust?"))

Loading weights: 100%|███████████████| 458/458 [00:01<00:00, 375.41it/s, Materializing param=model.vision_tower.transformer.layers.23.ffn_norm.weight]
Some parameters are on the meta device because they were offloaded to the disk and cpu.


During the **Holocaust (1939–1945)**, **Spain under Francisco Franco’s dictatorship** maintained a **neutral stance** in World War II but exhibited **complicity in antisemitic policies, forced deportations, and indirect support for Axis atrocities**. Here’s a detailed breakdown of Spain’s situation during the Holocaust:

---

### **1. Neutrality and Limited Engagement**
- **Official Neutrality**: Spain avoided direct involvement in the war, though it remained a **safe haven for some Jewish refugees** (unlike Nazi-occupied Europe).
- **Blue Division (División Azul)**: Around **30,000 Spanish volunteers** fought under Nazi Germany’s command in the **Soviet Union (1941–1944)**. While not directly involved in the Holocaust, their presence **legitimized Franco’s alliance with the Axis** and may have influenced later policies.
- **No Explicit Collaboration**: Unlike Nazi-occupied territories (e.g., France, Belgium), Spain **did not enforce racial laws** (e.g., Nuremberg Laws) or systematical

In [199]:
test2 = "Which archives contain Holocaust-relevant material in the Netherlands?"
print(get_results_llm(test2))
print(get_results_llm_with_rag(test2))

In the Netherlands, several archives and institutions hold significant Holocaust-relevant materials, including documents, photographs, testimonies, and personal records. Here are the key archives and collections:

### **1. National Archives of the Netherlands (Nationaal Archief)**
   - **Location:** The Hague
   - **Key Collections:**
     - **Archives of the Jewish Council (Joodse Raad)** – Records of the Jewish Council established by the Nazi occupiers.
     - **Wartime Administration Records** – Documents from the German occupation, including deportation lists.
     - **Postwar Investigations** – Files on war crimes and collaboration.
   - **Website:** [www.nationaalarchief.nl](https://www.nationaalarchief.nl)

### **2. Dutch Institute for War Documentation (NIOD)**
   - **Location:** Amsterdam
   - **Key Collections:**
     - **Holocaust and War Documentation** – Extensive archives on the persecution of Jews, resistance, and postwar trials.
     - **Testimonies and Oral Histories**

In [200]:
test3 = "Can you provide relevant bibliography for the Holocaust in Belgium?"
print(get_results_llm(test3))
print(get_results_llm_with_rag(test3))

Certainly! Below is a selection of relevant academic and historical works on the Holocaust in Belgium, covering topics such as persecution, resistance, collaboration, and memory.

### **General Overviews & Historical Accounts**
1. **Bauer, Yehuda.** *Jews for Sale: Nazi-Jewish Negotiations, 1933–1945.* (1994) – Includes discussions on Belgian Jewish communities.
2. **Hilberg, Raul.** *The Destruction of the European Jews.* (1961) – A foundational work with sections on Belgium.
3. **Lipphardt, Hans.** *The Drancy Internment Camp: A Memoir of the Holocaust.* (1992) – Covers Belgian deportees sent to Drancy.
4. **Megargee, Geoffrey P.** *The Holocaust in Eastern Europe: A History.* (2021) – Includes Belgian context.
5. **Pohl, Dieter.** *National Socialism and the Peoples of Europe.* (2000) – Discusses Nazi policies in Belgium.

### **Belgian-Specific Studies**
6. **Bauer, Yehuda.** *A History of the Holocaust.* (2001) – Includes Belgian case studies.
7. **De Wever, Bruno.** *De collabora

In [212]:
test4 = "Where is Beit Lohamei archive located?"
print(get_results_llm(test4))
print(get_results_llm_with_rag(test4))

The **Beit Lohamei HaGeta'ot (Ghetto Fighters' House)** is located in **Givat Chaim Ichud**, near **Kibbutz Lohamei HaGeta'ot**, in the **Western Galilee region of Israel**.

### **Exact Location:**
- **Address:** 19 HaNitzanim St., Givat Chaim Ichud, Israel
- **Nearest City:** Approximately **15 km (9 miles) northeast of Haifa** and about **30 km (18 miles) southwest of Acre (Akko)**.

### **About the Museum:**
The **Ghetto Fighters' House** is a museum and research center dedicated to preserving the history of the Jewish resistance during the Holocaust, particularly in ghettos and concentration camps. It was founded in 1949 by Holocaust survivors who established **Kibbutz Lohamei HaGeta'ot**.

The museum includes:
- **Exhibits on Jewish resistance** during the Holocaust.
- **Archives of survivor testimonies, documents, and artifacts.**
- **A memorial to the Warsaw Ghetto Uprising and other resistance movements.**

If you're planning a visit, it's recommended to check their official w